# Retrieval & Hybrid Data Access

Structured + unstructured retrieval, citations, and combining sources in one turn.

**Rollup: 29/29 cases passed** across 4 required capabilities, est. cost $0.0465, 2054s total.

This is an executive-level summary over already-captured real-LLM results -- see `docs/CAPABILITY_MAPPING.md` for the full 25-capability table, and `notebooks/capabilities/<NN>_<slug>/demo.ipynb` for every case in full detail per capability.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 10. Structured and unstructured data retrieval from multiple data sources

8/8 passed — $0.0125, 640s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("What is Corona's growth strategy outside Mexico?", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: What is Corona's growth strategy outside Mexico?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-008', 'DOC-011', 'DOC-014']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **Middle Americas** zone (which includes Mexico) instead, plus any Mexico-specific commentary found in the documents.

AGENT: Based on the retrieved evidence, I unfortunately **cannot give you a concrete articulation of Corona's growth strategy outside Mexico** — none of the retrieved documents contains an explicit strategy statement (e.g., premiumization, market-expansion priorities, or investment focus) for Corona internat

## 11. Document retrieval with source citations

7/7 passed — $0.0117, 522s

In [ ]:
_ = ask("What is Corona's growth strategy outside Mexico?", '01')
_ = ask("What's driving Brazil's volume decline?", '02')

01
USER: What is Corona's growth strategy outside Mexico?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-012', 'DOC-004', 'DOC-008', 'DOC-011', 'DOC-014']]
[assumptions/limitations surfaced:]
  - AB InBev doesn't publicly disclose structured financials by country; showing the **Middle Americas** zone (which includes Mexico) instead, plus any Mexico-specific commentary found in the documents.

AGENT: Based on the retrieved evidence, I unfortunately **cannot give you a concrete articulation of Corona's growth strategy outside Mexico** — none of the retrieved documents contains an explicit strategy statement (e.g., premiumization, market-expansion priorities, or investment focus) for Corona internationally.

Here is what the evidence **does** support:

- **Brand ownership context**: Outside Mexico, the Corona/Modelo brand rights are held by AB InBev (Constellation Brands holds a permanent license only for Mexico) [DOC-012].
- **Megabrand momentum**: Corona is one of 

## 12. Hybrid data retrieval

6/6 passed — $0.0117, 612s

In [ ]:
_ = ask('How did North America perform in Q1 2024, and what does the earnings commentary say about it?', '01')
_ = ask("What was AB InBev's revenue in 2025, and how does that compare to Heineken's public position?", '02')

01
USER: How did North America perform in Q1 2024, and what does the earnings commentary say about it?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-001', 'DOC-005', 'DOC-011', 'DOC-013', 'DOC-015']]

AGENT: ## North America — Q1 2024

**Headline metrics** (structured KPI data + commentary in [DOC-001]):

| Metric | Q1 2024 |
|---|---|
| Revenue | $3,593M |
| Organic revenue growth | **-8.8%** |
| Volume | 21,353K hL (~21.4M hL) |
| Normalized EBITDA | $1,126M |
| EBITDA margin | 31.3% (computed) |

Source: *AB InBev First Quarter 2024 Results* (BusinessWire, 7 May 2024).

**What the earnings commentary says**

The retrieved Q1 2024 zone-by-zone commentary [DOC-001] reports North America exactly in line with the structured data: revenue of $3,593M with **organic growth of -8.8%**, volume of 21,353K hL, and normalized EBITDA of $1,126M at a **31.3% margin**.

That is the full extent of the North America narrative in the retrieved material. Two import

## 23. Document filtering using metadata, tags, and recency

8/8 passed — $0.0107, 281s

In [ ]:
_ = ask('What are the most recent earnings documents mentioning Asia Pacific?', '01')
_ = ask('Show me documents related to North America', '02')

01
USER: What are the most recent earnings documents mentioning Asia Pacific?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-007', 'DOC-015', 'DOC-008', 'DOC-006', 'DOC-013']]

AGENT: The most recent earnings documents retrieved that could involve Asia Pacific are the quarterly/full-year earnings commentaries:

- **[DOC-008] FY2025 Full-Year Results and Q4 2025 Summary** (dated 2026-02-11) — the most recent earnings document.
- **[DOC-007] Q3 2025 Results: Zone-by-Zone Summary** (dated 2025-10-30).
- **[DOC-006] Q2 2025 Results: Zone-by-Zone Summary** (dated 2025-07-31) — earlier, less recent.

**Important caveat:** In the retrieved excerpts, these zone-by-zone summaries only show North America, Middle Americas, and South America before the text cuts off. No Asia Pacific figures are actually visible in the evidence provided, so I cannot confirm from the retrieved snippets which documents contain explicit Asia Pacific data — though the documents are structured as z